# Drift Detection — 05: Autoencoder (Anomaly Detection)

Trains a neural network to reconstruct "normal" alignment trajectories.
High reconstruction error = anomaly = potential drift.

**Doc reference:** `docs/evolution/drift_detection.md § 3.5`

### ⚠️ Data limitation

> With 24 annotated personas × ~10 entries each ≈ 240 trajectory windows,
> an autoencoder will **memorise rather than generalise**.
>
> This notebook is a proof-of-concept implementation. The approach becomes
> meaningful at ~500+ diverse trajectory windows (hundreds of real users
> with months of journaling history).

### Architecture

- **Input:** Window of W=4 steps × 10 dimensions = 40-dim vector
- **Encoder:** 40 → 20 → 10
- **Bottleneck:** 10-dim latent representation
- **Decoder:** 10 → 20 → 40
- **Loss:** MSE reconstruction loss on "aligned" training windows


In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=0.9)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

# --- Load judge labels (204 personas, 1651 entries, integer {-1, 0, 1} scores) ---
# Using judge labels rather than human annotations: more data (all 204 personas vs 24),
# and clean integer scores (no averaging artefacts).

VALUE_COLS = [
    "alignment_self_direction", "alignment_stimulation", "alignment_hedonism",
    "alignment_achievement", "alignment_power", "alignment_security",
    "alignment_conformity", "alignment_tradition", "alignment_benevolence",
    "alignment_universalism",
]
SHORT_NAMES = [c.replace("alignment_", "") for c in VALUE_COLS]
DIM_LABELS  = ["SD", "ST", "HE", "AC", "PO", "SE", "CO", "TR", "BE", "UN"]

judge_df = pl.read_parquet("../../logs/judge_labels/judge_labels.parquet")
mean_df  = (
    judge_df
    .select(["persona_id", "t_index"] + VALUE_COLS)
    .with_columns([pl.col(c).cast(pl.Float64) for c in VALUE_COLS])
    .sort(["persona_id", "t_index"])
)

registry   = pl.read_parquet("../../logs/registry/personas.parquet").select(
                ["persona_id", "name", "core_values"])
id_to_name = dict(zip(registry["persona_id"].to_list(), registry["name"].to_list()))
id_to_core = dict(zip(registry["persona_id"].to_list(), registry["core_values"].to_list()))

persona_ids = sorted(mean_df["persona_id"].unique().to_list())
print(f"Loaded {len(judge_df)} judge-labelled entries across {len(persona_ids)} personas")

# --- Sample: verify input scores look correct ---
# Pick first persona with ≥5 entries so the sample is representative
sample_pid = next(
    pid for pid in persona_ids
    if mean_df.filter(pl.col("persona_id") == pid).height >= 5
)
sample_data = mean_df.filter(pl.col("persona_id") == sample_pid).head(5)
print(f"\nSample scores (first 5 entries) for '{id_to_name.get(sample_pid, sample_pid)}':")
print(f"  Core values: {id_to_core.get(sample_pid, [])}")
print(sample_data.select(["t_index"] + VALUE_COLS))
print("  Scores are integers in {{-1, 0, +1}} across all 10 Schwartz dimensions")


def get_persona_matrix(pid: str) -> tuple[list[int], np.ndarray]:
    pdata = mean_df.filter(pl.col("persona_id") == pid).sort("t_index")
    return pdata["t_index"].to_list(), np.array([pdata[c].to_list() for c in VALUE_COLS]).T


def get_profile_weights(pid: str) -> np.ndarray:
    core = id_to_core.get(pid, [])
    name_to_idx = {s.lower().replace("-", "_"): i for i, s in enumerate(SHORT_NAMES)}
    w = np.zeros(10)
    for v in core:
        key = v.lower().replace("-", "_").replace(" ", "_")
        if key in name_to_idx:
            w[name_to_idx[key]] = 1.0
    if w.sum() > 0:
        w /= w.sum()
    return w


def core_dim_indices(weights: np.ndarray, w_min: float = 0.15) -> list[int]:
    return [j for j, wj in enumerate(weights) if wj >= w_min]


## Build Training Windows

Extract sliding windows of W steps from persona trajectories.
Train only on windows from the 'aligned' portion (first ~40% of each persona's trajectory,
assuming alignment holds early on).

In [ ]:
W     = 4    # window size
W_MIN = 0.15

def extract_windows(matrix: np.ndarray, window: int = W) -> np.ndarray:
    """Extract all sliding windows of size `window` from (T, K) matrix."""
    T, K = matrix.shape
    if T < window:
        return np.empty((0, window * K))
    return np.array([matrix[t:t+window].flatten() for t in range(T - window + 1)])


# Training set: first 40% of each persona (assumed aligned baseline)
train_windows = []
for pid in persona_ids:
    t_idx, matrix = get_persona_matrix(pid)
    T = len(t_idx)
    if T < W + 2:
        continue
    cutoff = max(W, int(T * 0.4))
    wins = extract_windows(matrix[:cutoff])
    train_windows.append(wins)

if train_windows:
    X_train = np.vstack(train_windows).astype(np.float32)
else:
    X_train = np.empty((0, W * 10), dtype=np.float32)

print(f"Training windows: {X_train.shape[0]} windows of size {W} steps × 10 dims")
print(f"WARNING: {X_train.shape[0]} windows is far below the ~500+ needed for generalisation.")
print("This notebook demonstrates the implementation — treat results as illustrative only.")


## Autoencoder Architecture

In [ ]:
class DriftAutoencoder(nn.Module):
    def __init__(self, input_dim: int = W * 10, latent_dim: int = 10):
        super().__init__()
        hidden = input_dim * 2
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, input_dim),
            nn.Tanh(),   # scores are in [-1, 1]
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


model_ae = DriftAutoencoder(input_dim=W * 10, latent_dim=10)
print(model_ae)
print(f"\nParameters: {sum(p.numel() for p in model_ae.parameters()):,}")


## Train

In [ ]:
EPOCHS    = 150
LR        = 1e-3
BATCH     = min(16, max(1, len(X_train) // 4))

recon_errors_train = []

if X_train.shape[0] >= 4:
    dataset = TensorDataset(torch.from_numpy(X_train))
    loader  = DataLoader(dataset, batch_size=BATCH, shuffle=True)

    optimiser = torch.optim.Adam(model_ae.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.MSELoss()

    model_ae.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for (x_batch,) in loader:
            optimiser.zero_grad()
            x_hat = model_ae(x_batch)
            loss  = criterion(x_hat, x_batch)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item() * len(x_batch)
        recon_errors_train.append(epoch_loss / len(X_train))

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(recon_errors_train)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.set_title("Autoencoder training loss (MSE)")
    plt.tight_layout()
    plt.show()
    print(f"Final training MSE: {recon_errors_train[-1]:.4f}")
else:
    print("Insufficient training data — skipping training.")


## Compute Reconstruction Error on Full Trajectories

Slide the window across each persona's full trajectory. High reconstruction error at step t
indicates the window ending at t looks different from training patterns.

In [ ]:
model_ae.eval()
AE_THRESH_QUANTILE = 0.90   # flag top 10% reconstruction errors as anomalies

# Compute errors on training set to set threshold
if X_train.shape[0] >= 4:
    with torch.no_grad():
        x_tensor = torch.from_numpy(X_train)
        recon    = model_ae(x_tensor).numpy()
    train_errors = ((X_train - recon) ** 2).mean(axis=1)
    threshold = float(np.quantile(train_errors, AE_THRESH_QUANTILE))
    print(f"Alert threshold (p{AE_THRESH_QUANTILE*100:.0f} of training errors): {threshold:.4f}")
else:
    threshold = 0.1
    print("Using default threshold (no training data)")

ae_results = {}

for pid in persona_ids:
    t_idx, matrix = get_persona_matrix(pid)
    T = len(t_idx)
    if T < W + 2:
        continue

    windows = extract_windows(matrix)
    if len(windows) == 0:
        continue
    x_t = torch.from_numpy(windows.astype(np.float32))

    with torch.no_grad():
        recon = model_ae(x_t).numpy()
    errors = ((windows - recon) ** 2).mean(axis=1)

    # t-step of each window = last step of the window
    window_t = list(range(W - 1, T))
    alerts = [(window_t[i], -1) for i, e in enumerate(errors) if e > threshold]

    ae_results[pid] = {
        "name":    id_to_name.get(pid, pid[:8]),
        "core":    id_to_core.get(pid, []),
        "T":       T,
        "t_idx":   t_idx,
        "matrix":  matrix,
        "weights": get_profile_weights(pid),
        "errors":  errors,
        "window_t": window_t,
        "alerts":  alerts,
        "threshold": threshold,
    }

print(f"\nAE anomaly detection on {len(ae_results)} personas:")
for pid, r in ae_results.items():
    print(f"  {r['name']:<25s}  alerts: {len(r['alerts'])}")


## Visualise Reconstruction Error

In [ ]:
palette = sns.color_palette("tab10", n_colors=10)


def plot_ae_persona(pid: str):
    r = ae_results[pid]
    matrix  = r["matrix"]
    T       = r["T"]
    errors  = r["errors"]
    wt      = r["window_t"]

    core_j = core_dim_indices(r["weights"], W_MIN)

    n_rows = len(core_j) + 1
    fig, axes = plt.subplots(n_rows, 1, figsize=(12, 2.5 * n_rows), sharex=True)

    # Score plots
    for ax, j in zip(axes[:-1], core_j):
        ax.plot(range(T), matrix[:, j], "o-", color=palette[j], lw=1.8)
        ax.axhline(0, color="grey", lw=0.5, ls="--")
        ax.set_ylim(-1.4, 1.4)
        ax.set_ylabel(DIM_LABELS[j], fontsize=9)
        for t, _ in r["alerts"]:
            ax.axvspan(t - 0.4, t + 0.4, color="red", alpha=0.2)

    # Reconstruction error
    ax_err = axes[-1]
    ax_err.bar(wt, errors, color="salmon", alpha=0.8, width=0.6)
    ax_err.axhline(r["threshold"], color="red", lw=1.2, ls="--", label=f"threshold={r['threshold']:.3f}")
    ax_err.set_ylabel("Recon. error", fontsize=9)
    ax_err.legend(fontsize=8)

    axes[0].set_title(f"{r['name']} — {', '.join(r['core'])} (⚠ memorisation risk at this data scale)",
                      fontweight="bold")
    axes[-1].set_xlabel("t_index")
    plt.tight_layout()
    plt.show()


for pid in list(ae_results.keys())[:4]:
    plot_ae_persona(pid)


## Limitations (Reiterated)

- **Data:** ~240 windows is far below the ~500+ needed. The autoencoder memorises training trajectories rather than learning general 'normal' patterns. Reconstruction errors will be low on training personas and unreliable on others.
- **No profile weights:** Reconstruction error is aggregate across all 10 dimensions. A misalignment on a low-weight dimension contributes equally to a misalignment on a core value.
- **Interpretability:** `error = 0.73` tells the Coach nothing about *which* value changed or *why*.
- **Cross-dimension patterns:** The autoencoder's main advantage — detecting correlated cross-dimension shifts (e.g., Self-Direction down + Achievement up) — requires much more data to be reliable.

**Recommendation:** Do not use autoencoders until Twinkl has hundreds of real users with months of history. Simpler cosine similarity already captures the directional component at this scale. See `docs/evolution/drift_detection.md § 3.5`.